
# Notebook 18 — Manuscript-Ready Results Package

## الهدف

تجميع مخرجات **Notebook 16** و**Notebook 17** في حزمة نهائية جاهزة للمخطوطة، تشمل:

- التحقق النهائي من الاتساق والجودة.
- جداول داخلية وخارجية جاهزة للنشر.
- رسوم بدقة 300 dpi.
- نصوص نتائج وتفسير علمي جاهزة للمراجعة.
- ملف Excel شامل.
- ملف Word مختصر للنتائج.
- قائمة تحقق نهائية قبل كتابة المخطوطة.

> هذا الدفتر لا يعيد تدريب أي نموذج ولا يغيّر أي عتبة.


In [ ]:

# Cell 1 — Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/PPMI_PD_Progression')

N16_DIR = (
    PROJECT_DIR
    / 'outputs'
    / 'notebook_16_harmonized_clinical_model_7_features'
)

N17_DIR = (
    PROJECT_DIR
    / 'outputs'
    / 'notebook_17_pdbp_external_validation'
)

OUTPUT_DIR = (
    PROJECT_DIR
    / 'outputs'
    / 'notebook_18_manuscript_ready_results'
)

TABLES_DIR = OUTPUT_DIR / 'tables'
FIGURES_DIR = OUTPUT_DIR / 'figures'
TEXT_DIR = OUTPUT_DIR / 'text'

for p in [OUTPUT_DIR, TABLES_DIR, FIGURES_DIR, TEXT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Notebook 16:', N16_DIR)
print('Notebook 17:', N17_DIR)
print('Notebook 18 output:', OUTPUT_DIR)

assert N16_DIR.exists(), f'Missing Notebook 16 outputs: {N16_DIR}'
assert N17_DIR.exists(), f'Missing Notebook 17 outputs: {N17_DIR}'


In [ ]:

# Cell 2 — Imports
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from docx import Document
from docx.shared import Inches, Pt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:

# Cell 3 — Utility functions
def read_csv_required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def read_text_required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path.read_text(encoding='utf-8')

def save_fig(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


In [ ]:

# Cell 4 — Load Notebook 16 outputs
n16_split = read_csv_required(
    N16_DIR / '03_train_test_split_summary.csv'
)

n16_internal = read_csv_required(
    N16_DIR / '05_harmonized_internal_test_performance.csv'
)

n16_missingness = read_csv_required(
    N16_DIR / '02_harmonized_predictor_missingness.csv'
)

n16_mapping = read_csv_required(
    N16_DIR / '01_harmonized_feature_mapping.csv'
)

n16_qc = read_csv_required(
    N16_DIR / '12_quality_control_checklist.csv'
)

n16_summary = read_text_required(
    N16_DIR / '13_notebook_16_summary_report.txt'
)

display(n16_internal)


In [ ]:

# Cell 5 — Load Notebook 17 outputs
n17_cohort = read_csv_required(
    N17_DIR / '02_pdbp_external_cohort_summary.csv'
)

n17_external = read_csv_required(
    N17_DIR / '04_pdbp_external_performance.csv'
)

n17_missingness = read_csv_required(
    N17_DIR / '01_pdbp_predictor_missingness.csv'
)

n17_comparison = read_csv_required(
    N17_DIR / '08_ppmi_internal_vs_pdbp_external_performance.csv'
)

n17_qc = read_csv_required(
    N17_DIR / '09_quality_control_checklist.csv'
)

n17_summary = read_text_required(
    N17_DIR / '10_notebook_17_summary_report.txt'
)

display(n17_external)


In [ ]:

# Cell 6 — Final consistency checks
qc_rows = []

def add_qc(check, status, detail):
    qc_rows.append({
        'check': check,
        'status': status,
        'detail': detail
    })

n16_all_pass = (n16_qc['status'].astype(str).str.upper() == 'PASS').all()
n17_all_pass = (n17_qc['status'].astype(str).str.upper() == 'PASS').all()

add_qc(
    'Notebook 16 QC complete',
    'PASS' if n16_all_pass else 'FAIL',
    f'{(n16_qc.status.str.upper()=="PASS").sum()}/{len(n16_qc)} PASS'
)

add_qc(
    'Notebook 17 QC complete',
    'PASS' if n17_all_pass else 'FAIL',
    f'{(n17_qc.status.str.upper()=="PASS").sum()}/{len(n17_qc)} PASS'
)

n_dev = int(n16_split['n'].sum())
n_ext = int(n17_cohort.iloc[0]['n'])

add_qc(
    'Development cohort size available',
    'PASS' if n_dev > 0 else 'FAIL',
    f'n={n_dev}'
)

add_qc(
    'External cohort size available',
    'PASS' if n_ext > 0 else 'FAIL',
    f'n={n_ext}'
)

add_qc(
    'Seven harmonized predictors confirmed',
    'PASS' if len(n16_mapping) == 7 else 'FAIL',
    f'n={len(n16_mapping)}'
)

final_qc = pd.DataFrame(qc_rows)
display(final_qc)

final_qc.to_csv(
    OUTPUT_DIR / '01_final_quality_control_checklist.csv',
    index=False
)


In [ ]:

# Cell 7 — Table 1: Development and external cohort summary
train_row = n16_split.loc[n16_split['split'] == 'train'].iloc[0]
test_row = n16_split.loc[n16_split['split'] == 'test'].iloc[0]
pdbp_row = n17_cohort.iloc[0]

table1 = pd.DataFrame([
    {
        'Cohort': 'PPMI development cohort',
        'Role': 'Model development and internal validation',
        'Participants': int(train_row['n'] + test_row['n']),
        'Rapid_progressors': int(
            train_row['rapid_progressors']
            + test_row['rapid_progressors']
        ),
        'Rapid_progressor_percent': round(
            100 * (
                train_row['rapid_progressors']
                + test_row['rapid_progressors']
            ) / (train_row['n'] + test_row['n']),
            2
        ),
        'Outcome_definition': (
            'Annualized MDS-UPDRS Part III change '
            '>= 5.0793 points/year'
        )
    },
    {
        'Cohort': 'PDBP external validation cohort',
        'Role': 'Independent external validation',
        'Participants': int(pdbp_row['n']),
        'Rapid_progressors': int(pdbp_row['rapid_progressors']),
        'Rapid_progressor_percent': round(
            float(pdbp_row['rapid_progressor_percent']),
            2
        ),
        'Outcome_definition': (
            'Annualized MDS-UPDRS Part III change '
            '>= 5.0793 points/year'
        )
    }
])

display(table1)
table1.to_csv(
    TABLES_DIR / 'Table_1_Cohort_Summary.csv',
    index=False
)


In [ ]:

# Cell 8 — Table 2: Harmonized predictor mapping
table2 = n16_mapping.copy()
table2.columns = [
    'PPMI_predictor',
    'Available_in_PPMI',
    'PDBP_mapping'
]

display(table2)

table2.to_csv(
    TABLES_DIR / 'Table_2_Harmonized_Predictor_Mapping.csv',
    index=False
)


In [ ]:

# Cell 9 — Table 3: Internal validation performance
table3 = n16_internal.copy()

table3.insert(0, 'Cohort', 'PPMI internal held-out test')
table3.insert(1, 'Validation_type', 'Internal validation')

display(table3)

table3.to_csv(
    TABLES_DIR / 'Table_3_Internal_Validation_Performance.csv',
    index=False
)


In [ ]:

# Cell 10 — Table 4: External validation performance
table4 = n17_external.copy()
table4.insert(0, 'Cohort', 'PDBP')

display(table4)

table4.to_csv(
    TABLES_DIR / 'Table_4_External_Validation_Performance.csv',
    index=False
)


In [ ]:

# Cell 11 — Table 5: Internal versus external comparison
table5 = n17_comparison.copy()

display(table5)

table5.to_csv(
    TABLES_DIR / 'Table_5_Internal_vs_External_Performance.csv',
    index=False
)


In [ ]:

# Cell 12 — Table 6: Missingness comparison
ppmi_miss = n16_missingness.copy()
ppmi_miss['Cohort'] = 'PPMI'

pdbp_miss = n17_missingness.copy()
pdbp_miss['Cohort'] = 'PDBP'

table6 = pd.concat(
    [ppmi_miss, pdbp_miss],
    ignore_index=True,
    sort=False
)

display(table6)

table6.to_csv(
    TABLES_DIR / 'Table_6_Predictor_Missingness_by_Cohort.csv',
    index=False
)


In [ ]:

# Cell 13 — Figure 1: Study workflow
plt.figure(figsize=(12, 3))
plt.axis('off')

boxes = [
    (0.05, 0.35, 0.16, 0.35, 'PPMI\nDevelopment cohort\n(n=856)'),
    (0.27, 0.35, 0.16, 0.35, '7 harmonized\nclinical predictors'),
    (0.49, 0.35, 0.16, 0.35, 'Locked PPMI\nlogistic model'),
    (0.71, 0.35, 0.16, 0.35, 'PDBP\nExternal validation\n(n=349)')
]

for x, y, w, h, label in boxes:
    rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.5)
    plt.gca().add_patch(rect)
    plt.text(
        x + w/2,
        y + h/2,
        label,
        ha='center',
        va='center'
    )

for i in range(len(boxes)-1):
    x1 = boxes[i][0] + boxes[i][2]
    y1 = boxes[i][1] + boxes[i][3]/2
    x2 = boxes[i+1][0]
    y2 = boxes[i+1][1] + boxes[i+1][3]/2
    plt.annotate(
        '',
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle='->', linewidth=1.5)
    )

plt.title(
    'Study Workflow: Development in PPMI and External Validation in PDBP',
    pad=10
)

save_fig('Figure_1_Study_Workflow.png')


In [ ]:

# Cell 14 — Figure 2: Outcome prevalence comparison
plot_df = table1[['Cohort', 'Rapid_progressor_percent']].copy()

plt.figure(figsize=(8, 5))
plt.bar(
    plot_df['Cohort'],
    plot_df['Rapid_progressor_percent']
)

plt.ylabel('Rapid progressors (%)')
plt.title('Rapid Motor Progression Prevalence by Cohort')
plt.xticks(rotation=15, ha='right')

for i, v in enumerate(plot_df['Rapid_progressor_percent']):
    plt.text(i, v + 0.5, f'{v:.1f}%', ha='center')

save_fig('Figure_2_Outcome_Prevalence_Comparison.png')


In [ ]:

# Cell 15 — Figure 3: Internal vs external performance
metrics = [
    'ROC_AUC',
    'PR_AUC',
    'balanced_accuracy',
    'sensitivity',
    'specificity',
    'F1'
]

plot_data = (
    table5
    .set_index('cohort')[metrics]
    .T
)

plot_data.plot(
    kind='bar',
    figsize=(11, 6)
)

plt.ylim(0, 1)
plt.ylabel('Metric value')
plt.xlabel('Performance metric')
plt.title('Internal and External Validation Performance')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Cohort')

save_fig('Figure_3_Internal_vs_External_Performance.png')


In [ ]:

# Cell 16 — Figure 4: Generalization drop
internal_row = table5.loc[
    table5['cohort'].str.contains(
        'PPMI',
        case=False,
        na=False
    )
].iloc[0]

external_row = table5.loc[
    table5['cohort'].str.contains(
        'PDBP',
        case=False,
        na=False
    )
].iloc[0]

delta = pd.DataFrame({
    'Metric': metrics,
    'External_minus_internal': [
        external_row[m] - internal_row[m]
        for m in metrics
    ]
})

display(delta)

plt.figure(figsize=(9, 5))
plt.bar(
    delta['Metric'],
    delta['External_minus_internal']
)
plt.axhline(0, linewidth=1)
plt.ylabel('External minus internal performance')
plt.title('Change in Performance During External Validation')
plt.xticks(rotation=30, ha='right')

save_fig('Figure_4_External_Generalization_Drop.png')

delta.to_csv(
    TABLES_DIR / 'Table_7_External_Generalization_Delta.csv',
    index=False
)


In [ ]:

# Cell 17 — Copy key Notebook 17 figures into Notebook 18 package
source_figures = [
    'Figure_1_PDBP_external_ROC_curve.png',
    'Figure_2_PDBP_external_precision_recall_curve.png',
    'Figure_3_PDBP_external_calibration_plot.png',
    'Figure_4_PDBP_predicted_probability_distribution.png',
    'Figure_5_PPMI_internal_vs_PDBP_external_performance.png'
]

copied = []

for name in source_figures:
    src = N17_DIR / name
    dst = FIGURES_DIR / name
    if src.exists():
        dst.write_bytes(src.read_bytes())
        copied.append(name)

print('Copied figures:')
print(copied)


In [ ]:

# Cell 18 — Manuscript-ready Results text
internal = table3.iloc[0]
external_primary = table4.loc[
    table4['analysis'] == 'Primary external validation'
].iloc[0]

results_text = f'''
Results

The harmonized development cohort included 856 participants from PPMI,
of whom 215 (25.12%) met the prespecified definition of rapid motor
progression. The independent PDBP external-validation cohort included
349 participants, of whom 54 (15.47%) were rapid progressors.

A seven-predictor harmonized clinical model was trained in PPMI using
age at enrollment, baseline MDS-UPDRS Part III, years since diagnosis,
MDS-UPDRS Part II, MDS-UPDRS Part I, MoCA total score, and Hoehn and
Yahr stage.

On the internal held-out PPMI test set, the model achieved a ROC-AUC of
{internal["ROC_AUC"]:.3f}, PR-AUC of {internal["PR_AUC"]:.3f}, balanced
accuracy of {internal["balanced_accuracy"]:.3f}, sensitivity of
{internal["sensitivity"]:.3f}, specificity of
{internal["specificity"]:.3f}, and F1-score of {internal["F1"]:.3f}.

When the locked PPMI pipeline was applied to PDBP without retraining,
feature reselection, preprocessing refitting, or threshold optimization,
external discrimination was limited. At the prespecified threshold of
0.50, ROC-AUC was {external_primary["ROC_AUC"]:.3f}, PR-AUC was
{external_primary["PR_AUC"]:.3f}, balanced accuracy was
{external_primary["balanced_accuracy"]:.3f}, sensitivity was
{external_primary["sensitivity"]:.3f}, specificity was
{external_primary["specificity"]:.3f}, precision was
{external_primary["precision"]:.3f}, and F1-score was
{external_primary["F1"]:.3f}. These findings indicate poor transportability
of the baseline harmonized clinical model across cohorts.
'''.strip()

print(results_text)

(TEXT_DIR / '01_Manuscript_Ready_Results.txt').write_text(
    results_text,
    encoding='utf-8'
)


In [ ]:

# Cell 19 — Manuscript-ready interpretation and discussion text
discussion_text = '''
Interpretation

The external validation showed that the harmonized baseline clinical
model did not generalize adequately from PPMI to PDBP. The external
ROC-AUC was close to chance and both PR-AUC and balanced accuracy were
low. Specificity remained relatively high at the 0.50 threshold, but
sensitivity was poor, indicating that most rapid progressors were not
identified.

This result should be interpreted as evidence of limited transportability,
not as a failure of the analytic workflow. The model was evaluated under
a strict external-validation design: PDBP was not used for training,
feature selection, preprocessing, model selection, or threshold
optimization. The performance decline therefore provides a realistic
estimate of cross-cohort generalization.

Potential explanations include differences between PPMI and PDBP in
participant selection, disease duration, clinical severity, medication
state, assessment protocols, missingness patterns, and event prevalence.
The lower prevalence of rapid progression in PDBP may also have
contributed to the reduced precision–recall performance.

The findings support three conclusions. First, baseline clinical
variables alone are insufficient for reliable prediction of rapid motor
progression across independent Parkinson's disease cohorts. Second,
internal validation performance should not be interpreted as evidence
of clinical readiness. Third, future work should prioritize longitudinal
predictors, cohort harmonization, recalibration, digital biomarkers,
genetic features, and prospectively defined multi-cohort model
development.
'''.strip()

print(discussion_text)

(TEXT_DIR / '02_Manuscript_Ready_Interpretation.txt').write_text(
    discussion_text,
    encoding='utf-8'
)


In [ ]:

# Cell 20 — Strengths, limitations, and future work
strengths = [
    'Independent PDBP external validation.',
    'Identical outcome threshold in PPMI and PDBP.',
    'Locked model and preprocessing pipeline.',
    'No feature reselection or threshold tuning on PDBP.',
    'Transparent multi-metric performance reporting.',
    'Reproducible notebook-based workflow.'
]

limitations = [
    'Only seven predictors were fully harmonized across cohorts.',
    'The harmonized model had modest internal discrimination.',
    'Differences in cohort design and assessment protocols may remain.',
    'No external recalibration was performed in the primary analysis.',
    'The outcome was based on two time points.',
    'Medication-state harmonization may be incomplete.',
    'The external event count was limited to 54 rapid progressors.'
]

future_work = [
    'Develop models using pooled multi-cohort training.',
    'Evaluate longitudinal and dynamic predictors.',
    'Add digital, imaging, genetic, and fluid biomarkers.',
    'Assess recalibration separately from pure external validation.',
    'Perform decision-curve analysis after adequate calibration.',
    'Validate prospectively in clinical settings.'
]

summary_table = pd.DataFrame({
    'Category': (
        ['Strength'] * len(strengths)
        + ['Limitation'] * len(limitations)
        + ['Future work'] * len(future_work)
    ),
    'Item': strengths + limitations + future_work
})

display(summary_table)

summary_table.to_csv(
    TABLES_DIR / 'Table_8_Strengths_Limitations_Future_Work.csv',
    index=False
)


In [ ]:

# Cell 21 — Export all tables to one Excel workbook
excel_path = OUTPUT_DIR / 'Manuscript_Tables.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    table1.to_excel(writer, sheet_name='Table1_Cohorts', index=False)
    table2.to_excel(writer, sheet_name='Table2_Predictors', index=False)
    table3.to_excel(writer, sheet_name='Table3_Internal', index=False)
    table4.to_excel(writer, sheet_name='Table4_External', index=False)
    table5.to_excel(writer, sheet_name='Table5_Comparison', index=False)
    table6.to_excel(writer, sheet_name='Table6_Missingness', index=False)
    delta.to_excel(writer, sheet_name='Table7_Deltas', index=False)
    summary_table.to_excel(
        writer,
        sheet_name='Table8_Interpretation',
        index=False
    )

print('Saved:', excel_path)


In [ ]:

# Cell 22 — Create Word results summary
doc = Document()

doc.add_heading(
    'PPMI–PDBP Harmonized Model: Manuscript-Ready Results',
    level=0
)

doc.add_heading('Key Results', level=1)
for paragraph in results_text.split('\n\n'):
    doc.add_paragraph(paragraph)

doc.add_heading('Interpretation', level=1)
for paragraph in discussion_text.split('\n\n'):
    doc.add_paragraph(paragraph)

doc.add_heading('Strengths', level=1)
for item in strengths:
    doc.add_paragraph(item, style='List Bullet')

doc.add_heading('Limitations', level=1)
for item in limitations:
    doc.add_paragraph(item, style='List Bullet')

doc.add_heading('Future Work', level=1)
for item in future_work:
    doc.add_paragraph(item, style='List Bullet')

doc.add_heading('Selected Figures', level=1)

for fig_name in [
    'Figure_1_Study_Workflow.png',
    'Figure_2_Outcome_Prevalence_Comparison.png',
    'Figure_3_Internal_vs_External_Performance.png',
    'Figure_1_PDBP_external_ROC_curve.png',
    'Figure_3_PDBP_external_calibration_plot.png'
]:
    fig_path = FIGURES_DIR / fig_name
    if fig_path.exists():
        doc.add_paragraph(fig_name)
        doc.add_picture(str(fig_path), width=Inches(6.2))

word_path = OUTPUT_DIR / 'Results_Summary.docx'
doc.save(word_path)

print('Saved:', word_path)


In [ ]:

# Cell 23 — Publication readiness checklist
checklist = pd.DataFrame([
    {
        'Item': 'Outcome definition identical across cohorts',
        'Status': 'Complete',
        'Notes': 'Annualized MDS-UPDRS III change >= 5.0793/year'
    },
    {
        'Item': 'Predictor harmonization documented',
        'Status': 'Complete',
        'Notes': 'Seven predictors'
    },
    {
        'Item': 'Model locked before external validation',
        'Status': 'Complete',
        'Notes': 'Notebook 16 pipeline'
    },
    {
        'Item': 'No external model tuning',
        'Status': 'Complete',
        'Notes': 'PDBP used only for evaluation'
    },
    {
        'Item': 'Internal performance reported',
        'Status': 'Complete',
        'Notes': 'PPMI held-out test'
    },
    {
        'Item': 'External performance reported',
        'Status': 'Complete',
        'Notes': 'PDBP'
    },
    {
        'Item': 'Calibration reported',
        'Status': 'Complete',
        'Notes': 'Calibration plot and Brier score'
    },
    {
        'Item': 'Decision-curve analysis',
        'Status': 'Not performed',
        'Notes': 'Not appropriate before adequate calibration'
    },
    {
        'Item': 'TRIPOD+AI checklist',
        'Status': 'Pending manuscript review',
        'Notes': ''
    },
    {
        'Item': 'PPMI publication review',
        'Status': 'Pending before journal submission',
        'Notes': ''
    },
    {
        'Item': 'AMP PDRD acknowledgement',
        'Status': 'Pending manuscript insertion',
        'Notes': ''
    }
])

display(checklist)

checklist.to_csv(
    OUTPUT_DIR / 'Publication_Readiness_Checklist.csv',
    index=False
)


In [ ]:

# Cell 24 — Final summary report
report_lines = [
    'Notebook 18 — Manuscript-Ready Results Package',
    '',
    f'PPMI development cohort: {table1.iloc[0]["Participants"]}',
    f'PDBP external cohort: {table1.iloc[1]["Participants"]}',
    f'Harmonized predictors: {len(table2)}',
    '',
    f'Internal ROC-AUC: {internal["ROC_AUC"]:.4f}',
    f'External ROC-AUC: {external_primary["ROC_AUC"]:.4f}',
    f'Internal PR-AUC: {internal["PR_AUC"]:.4f}',
    f'External PR-AUC: {external_primary["PR_AUC"]:.4f}',
    '',
    'Scientific conclusion:',
    (
        'The locked seven-predictor clinical model showed poor '
        'transportability from PPMI to PDBP.'
    ),
    '',
    'Files created:',
    str(excel_path),
    str(word_path),
    str(OUTPUT_DIR / 'Publication_Readiness_Checklist.csv'),
    '',
    f'Tables directory: {TABLES_DIR}',
    f'Figures directory: {FIGURES_DIR}',
    f'Text directory: {TEXT_DIR}'
]

report_text = '\n'.join(report_lines)
print(report_text)

(OUTPUT_DIR / 'Notebook_18_Final_Summary_Report.txt').write_text(
    report_text,
    encoding='utf-8'
)



## المخرجات النهائية

ستجد داخل:

```text
PPMI_PD_Progression/outputs/notebook_18_manuscript_ready_results/
```

الملفات الأساسية:

- `Manuscript_Tables.xlsx`
- `Results_Summary.docx`
- `Publication_Readiness_Checklist.csv`
- `Notebook_18_Final_Summary_Report.txt`
- مجلد `tables/`
- مجلد `figures/`
- مجلد `text/`

بعد نجاح هذا الدفتر يكون التحليل الإحصائي للمشروع مكتملًا.
